# 14 N-glycan Logistic Regression

## Purpose
This notebook asks a narrower, more controlled question than the full classifier notebooks:

- if we freeze one matched set of saved embedding spaces
- and fit only a simple logistic-regression probe
- how well do those embedding spaces separate `N-glycan` rows from other glycans?

This makes the downstream model intentionally simple so the comparison emphasizes the embedding geometry rather than classifier capacity.


## Setup note

This notebook is closer to notebook `11b` than notebook `10`:

- code stays in GitHub
- prepared classification tables and checkpoints stay in Drive
- embeddings are recomputed from saved checkpoints
- the only trained model inside this notebook is a small sklearn logistic regression

The intended use is a three-way comparison inside one tokenizer family plus one pretrained experiment, using the model states `pretrained_mlm`, `classification_mlm_init`, and `classification_random_init`.


## Runtime setup

This cell mounts Google Drive, synchronizes the repository, and installs the lightweight packages needed for embedding extraction plus the logistic-regression probe.

**Expected output**
- confirmation that Drive is mounted
- confirmation that the repository was cloned or updated
- the active repository directory in Colab


In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')

# Keep tqdm in plain-text mode for cleaner notebook logs.
from tqdm.std import tqdm as plain_tqdm
import tqdm.auto as tqdm_auto
tqdm_auto.tqdm = plain_tqdm
try:
    import tqdm.notebook as tqdm_notebook
    tqdm_notebook.tqdm = plain_tqdm
except Exception:
    pass

!pip install -q transformers scikit-learn

GITHUB_OWNER = "hb791-dev"
REPO_NAME = "glycan-roberta"
GITHUB_REF = "main"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git"
REPO_DIR = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Reusing existing repo at {REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", GITHUB_REF], check=True)

%cd {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Active repo directory: {REPO_DIR}")


## Import notebook helpers

This notebook keeps the repeatable mechanics in `src/classification_embedding_logreg.py` so the user-facing cells can stay focused on run selection and interpretation.

We run this cell early so the rest of the notebook can call a small set of readable helper functions instead of embedding the full logistic-regression workflow inline.

**Expected output**
- no printed output under normal conditions
- a standard Python import error only if the runtime setup step failed or a required package is missing

**How to interpret the result**
- if this cell runs quietly, the notebook is ready to build the run list and load the prepared classification tables
- if an import fails, fix the setup step before editing the later analysis cells


In [ ]:
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display

import src.classification_embedding_logreg as classification_embedding_logreg
importlib.reload(classification_embedding_logreg)

from src.classification_embedding_logreg import (
    MODEL_VARIANT_LABELS,
    build_classifier_run_label_resolution_table,
    build_embedding_logreg_run_config,
    build_embedding_logreg_output_paths,
    build_run_manifest,
    build_registry_run_specs,
    export_public_embedding_logreg_html,
    load_run_registry,
    prepare_n_glycan_probe_dataframe,
    resolve_classifier_run_label_choices,
    resolve_run_registry_path,
    summarize_main_glycan_class_by_split,
    run_embedding_logreg_suite,
    save_embedding_logreg_run_config,
)
from src.notebook_utils import SUPPORTED_TOKENIZER_FAMILIES, validate_tokenizer_family
from src.similarity import build_public_export_dir, build_public_report_subdir


## User settings

This is the main cell to edit before running the notebook.

**What to choose here**
- which tokenizer family and exact pretrained experiment to compare
- which pooling rule to use for the frozen sequence embeddings
- which encoder hidden-state layer should provide the embedding vectors
- which saved classifier run-label candidates should be checked for the two fine-tuned model states
- whether unlabeled rows should be kept as part of the non-`N-glycan` class
- the regularization and threshold settings for the logistic regression
- whether to also copy a clean public HTML folder after the report is created

This notebook is intentionally narrow: one tokenizer family plus one exact pretrained experiment expands into three related model states for the saved comparison report.

**Expected output**
- this cell only defines notebook settings; it does not run the comparison yet

**How to interpret the result**
- fill `EXPERIMENT_NAME` with the exact pretrained run you want to probe
- keep `MODEL_VARIANTS` as the default three-state tuple unless you are debugging a missing checkpoint
- use `EMBEDDING_LAYER_INDEX = -1` for the final layer, `-2` for the penultimate layer, or `0` for token embeddings before encoder blocks
- adjust the classifier run-label candidate tuples if the helper still does not find the saved fine-tuned checkpoints in Drive
- keep `TRAIN_SPLITS = ("train",)` and `EVALUATION_SPLITS = ("train", "val", "test")` so the notebook stays aligned with the standard project split names
- leave `PUBLIC_EXPORT_ENABLED = False` until the local Drive report looks correct, then turn it on to prepare the public-ready folder


In [ ]:
from pathlib import Path

# Update DRIVE_ROOT if the project folder uses a different Google Drive path.
DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CLASSIFICATION_PREP_DIR = DRIVE_ROOT / "results" / "classification_prep"
CHECKPOINTS_DIR = DRIVE_ROOT / "checkpoints"
RUN_REGISTRY_PATH = resolve_run_registry_path(drive_root=DRIVE_ROOT, repo_dir=REPO_DIR)

# Choose one tokenizer family and one exact pretrained experiment name.
# The notebook will expand that single experiment into the three saved
# model states listed below.
TOKENIZER_FAMILY = "manual"
EXPERIMENT_NAME = "mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only"
MODEL_VARIANTS = (
    "pretrained_mlm",
    "classification_mlm_init",
    "classification_random_init",
)

# The helper will try these saved folder names in order and use the first
# one that actually exists for this experiment in Drive.
CLASSIFIER_MLM_RUN_LABEL_CANDIDATES = (
    "cls_lr2e-5_ep100_bs16_mlm",
    "cls_lr2e-5_ep10_bs16_mlm",
)
CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES = (
    "cls_lr2e-5_ep100_bs16_randominit",
    "cls_lr2e-5_ep10_bs16_randominit",
    "cls_lr2e-5_ep10_bs16_random_init",
)

# Embedding and binary-probe settings.
POOLING_STRATEGY = "mean"
EMBEDDING_LAYER_INDEX = -1
SPLITS_TO_INCLUDE = ("train", "val", "test")
# Notebook 14 expects the standard project split names: train, val, and test.
# Fit on train, compare settings on val, and keep test as the locked final view.
TRAIN_SPLITS = ("train",)
EVALUATION_SPLITS = ("train", "val", "test")
# Current default: keep unlabeled rows as valid non-N-glycan examples.
EXCLUDE_UNLABELED_ROWS = False
PROBABILITY_THRESHOLD = 0.5
LOGREG_C = 1.0
CLASS_WEIGHT = "balanced"
MAX_ITER = 2000
RANDOM_STATE = 42
BATCH_SIZE = 32
MAX_LENGTH = None

# Output and overwrite behavior.
HTML_REPORT_TITLE = "N-glycan logistic-regression comparison"
COMPARISON_RUN_LABEL = "n_glycan_variant_comparison_mean_pool"
OVERWRITE_EXISTING_OUTPUTS = True
FAIL_ON_MISSING_MODEL_DIRS = False
METRIC_PLOT_NAMES = ("roc_auc", "average_precision", "f1", "balanced_accuracy")

# Configure the optional clean public-export folder.
PUBLIC_EXPORT_ENABLED = False
PUBLIC_EXPORT_PARENT_SUBDIR = 'results/public_reports'
PUBLIC_GITHUB_OWNER = 'hb791-dev'
PUBLIC_GITHUB_REPO = 'glycan-roberta'
PUBLIC_GITHUB_REF = 'main'
PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH = True

validate_tokenizer_family(TOKENIZER_FAMILY, supported_families=SUPPORTED_TOKENIZER_FAMILIES)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Classification prep dir: {CLASSIFICATION_PREP_DIR}")
print(f"Checkpoints dir: {CHECKPOINTS_DIR}")
print(f"Run registry path: {RUN_REGISTRY_PATH}")
print(f"Tokenizer family: {TOKENIZER_FAMILY}")
print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Model variants: {MODEL_VARIANTS}")
print(f"Classifier, MLM init candidates: {CLASSIFIER_MLM_RUN_LABEL_CANDIDATES}")
print(f"Classifier, random-init candidates: {CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES}")
print(f"Pooling strategy: {POOLING_STRATEGY}")
print(f"Embedding layer index: {EMBEDDING_LAYER_INDEX}")
print(f"Keep unlabeled rows: {not EXCLUDE_UNLABELED_ROWS}")
print(f"HTML report title: {HTML_REPORT_TITLE}")
print(f"Comparison run label: {COMPARISON_RUN_LABEL}")
print(f"Public export enabled: {PUBLIC_EXPORT_ENABLED}")


## Build the three-run comparison from the cleaned registry

This step finds one exact pretrained experiment in the cleaned registry, resolves the saved fine-tuned checkpoint folder names that actually exist, expands the comparison into the requested model states, and records the exact checkpoint folders that the probe will use.

**Expected output**
- a compact manifest of compared runs
- a count of how many requested model directories currently exist in Drive
- one saved config JSON in the notebook-14 output folder

**How to interpret the result**
- if the cell raises an error about multiple matched experiments, narrow `EXPERIMENT_NAME` until the manifest represents one architecture only
- the classifier run-label resolution table shows which saved fine-tuning folders were actually selected from the candidate lists
- the manifest is the main place to confirm that tokenizer family, architecture label, and model-state labels match the intended comparison
- the printed public-export paths show where the clean shareable HTML folder will be copied if that option is enabled


In [ ]:
# Load the cleaned registry so the comparison can reuse the existing run
# metadata instead of hardcoding checkpoint folders.
run_registry_df = load_run_registry(RUN_REGISTRY_PATH)
classifier_label_resolution = resolve_classifier_run_label_choices(
    checkpoints_dir=CHECKPOINTS_DIR,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    classifier_mlm_run_label_candidates=CLASSIFIER_MLM_RUN_LABEL_CANDIDATES,
    classifier_random_run_label_candidates=CLASSIFIER_RANDOM_RUN_LABEL_CANDIDATES,
)
classifier_label_resolution_df = build_classifier_run_label_resolution_table(
    classifier_label_resolution,
)
classifier_mlm_run_label = classifier_label_resolution["classification_mlm_init"]["selected_run_label"]
classifier_random_run_label = classifier_label_resolution["classification_random_init"]["selected_run_label"]

run_specs = build_registry_run_specs(
    run_registry_df,
    tokenizer_families=[TOKENIZER_FAMILY],
    experiment_names=[EXPERIMENT_NAME],
    model_variants=MODEL_VARIANTS,
    classifier_mlm_run_label=classifier_mlm_run_label,
    classifier_random_run_label=classifier_random_run_label,
)

if not run_specs:
    raise ValueError("No run specs were generated. Adjust TOKENIZER_FAMILY or EXPERIMENT_NAME.")

comparison_group_df = pd.DataFrame(run_specs)[
    ["tokenizer_family", "experiment_name", "architecture_label"]
].drop_duplicates()
if len(comparison_group_df) != 1:
    raise ValueError(
        "Notebook 14 is now scoped to one tokenizer family plus one exact architecture/experiment at a time. "
        "Refine EXPERIMENT_NAME until only one group remains."
    )

# Build the notebook-14 output folder and one manifest row per requested run.
output_paths = build_embedding_logreg_output_paths(
    DRIVE_ROOT,
    tokenizer_family=TOKENIZER_FAMILY,
    experiment_name=EXPERIMENT_NAME,
    comparison_run_label=COMPARISON_RUN_LABEL,
)
PUBLIC_REPORT_NOTEBOOK_STEM = '14_n_glycan_logistic_regression'
PUBLIC_EXPORT_PATH_PARTS = [TOKENIZER_FAMILY, EXPERIMENT_NAME, COMPARISON_RUN_LABEL]
PUBLIC_EXPORT_PARENT_DIR = DRIVE_ROOT / PUBLIC_EXPORT_PARENT_SUBDIR
PUBLIC_EXPORT_REPO_SUBDIR = build_public_report_subdir(
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
PUBLIC_EXPORT_DIR = build_public_export_dir(
    PUBLIC_EXPORT_PARENT_DIR,
    PUBLIC_REPORT_NOTEBOOK_STEM,
    PUBLIC_EXPORT_PATH_PARTS,
)
run_manifest_df = build_run_manifest(run_specs, checkpoints_dir=CHECKPOINTS_DIR)

# Save the active notebook settings so the exact comparison can be audited
# later without reopening the notebook itself.
run_config = build_embedding_logreg_run_config(
    drive_root=DRIVE_ROOT,
    classification_prep_dir=CLASSIFICATION_PREP_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    pooling_strategy=POOLING_STRATEGY,
    embedding_layer_index=EMBEDDING_LAYER_INDEX,
    splits_to_include=SPLITS_TO_INCLUDE,
    train_splits=TRAIN_SPLITS,
    evaluation_splits=EVALUATION_SPLITS,
    exclude_unlabeled_rows=EXCLUDE_UNLABELED_ROWS,
    probability_threshold=PROBABILITY_THRESHOLD,
    logreg_c=LOGREG_C,
    class_weight=CLASS_WEIGHT,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    comparison_run_label=COMPARISON_RUN_LABEL,
    output_paths=output_paths,
    run_specs=run_specs,
)
run_config['public_export'] = {
    'enabled': bool(PUBLIC_EXPORT_ENABLED),
    'public_export_parent_subdir': PUBLIC_EXPORT_PARENT_SUBDIR,
    'public_export_dir': str(PUBLIC_EXPORT_DIR),
    'public_export_repo_subdir': PUBLIC_EXPORT_REPO_SUBDIR,
    'public_github_owner': PUBLIC_GITHUB_OWNER,
    'public_github_repo': PUBLIC_GITHUB_REPO,
    'public_github_ref': PUBLIC_GITHUB_REF,
    'public_export_fail_on_sensitive_match': bool(PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH),
}
run_config['classifier_run_label_resolution'] = classifier_label_resolution_df.to_dict(orient='records')
save_embedding_logreg_run_config(output_paths["run_config_path"], run_config)

display_columns = [
    "display_label",
    "tokenizer_family",
    "architecture_label",
    "model_variant",
    "model_dir_exists",
]
print("Classifier run-label resolution")
display(classifier_label_resolution_df)
print("Requested run manifest")
display(run_manifest_df[display_columns])
print(f"Requested runs: {len(run_manifest_df)}")
print(f"Existing model dirs: {int(run_manifest_df['model_dir_exists'].sum())}")
print(f"Output dir: {output_paths['results_dir']}")
print(f"HTML report path: {output_paths['html_report_path']}")
print(f"Public export Drive folder: {PUBLIC_EXPORT_DIR}")
print(f"Repo destination after copy: {PUBLIC_EXPORT_REPO_SUBDIR}")


## Load notebook-09 outputs and build the binary target

This step reuses the prepared classification tables, derives the broader glycan-class metadata, and then collapses the labels to one binary target:

- positive class: `N-glycan`
- negative class: everything that is not `N-glycan`, including unlabeled rows when they are kept in the run

The split logic stays aligned with notebook 09: fit the logistic regression on `train`, use `val` to compare settings such as pooling or regularization, and keep `test` as the final locked comparison.

**Expected output**
- the number of rows kept for the probe
- a split-by-target count table for the binary `N-glycan` versus non-`N-glycan` view
- a broader glycan-class count table for a quick sanity check

**How to interpret the result**
- if the positive class is extremely small in one split, the later probe metrics may look unstable or noisy
- if the broad glycan-class counts look surprising, revisit the prepared classification tables before trusting the logistic-regression results


In [ ]:
# Load the prepared classification rows from notebook 09 and collapse the
# broader glycan-class metadata into one binary N-glycan probe target.
annotated_probe_df, label_vocabulary_df = prepare_n_glycan_probe_dataframe(
    train_csv_path=CLASSIFICATION_PREP_DIR / "train_classification.csv",
    val_csv_path=CLASSIFICATION_PREP_DIR / "val_classification.csv",
    test_csv_path=CLASSIFICATION_PREP_DIR / "test_classification.csv",
    label_vocabulary_path=CLASSIFICATION_PREP_DIR / "label_vocabulary.csv",
    splits_to_include=SPLITS_TO_INCLUDE,
    exclude_unlabeled_rows=EXCLUDE_UNLABELED_ROWS,
)

# Summarize the resulting binary target distribution so class balance can be
# reviewed before the notebook starts embedding and fitting probes.
target_summary_df = classification_embedding_logreg.summarize_binary_target(annotated_probe_df)
class_summary_df = summarize_main_glycan_class_by_split(annotated_probe_df)

print(f"Rows kept for the probe: {len(annotated_probe_df)}")
print("Binary target counts by split")
display(target_summary_df)
print("Broad glycan class counts")
display(class_summary_df)


## Run the embedding probe suite

For each requested run, the notebook will:

- load the saved checkpoint
- embed the glycan sequences with the selected pooling rule
- fit one logistic regression on the train split only
- evaluate that same probe on train, validation, and test
- save per-run predictions, metric summary grids, full logistic-regression diagnostics, and one comparison HTML report
- optionally copy a clean public-ready HTML folder for later GitHub publishing

**Expected output**
- a count of completed runs
- a count of skipped runs whose checkpoint folders were missing
- the main saved output paths for the metrics table, test summary, and HTML report
- saved ROC, precision-recall, confusion-matrix, and probability-distribution plots for validation and test
- when public export is enabled, copied-file and scan tables for the clean public folder

**How to interpret the result**
- this step is the longest part of the notebook because it recomputes embeddings from saved checkpoints
- skipped runs are not automatically fatal unless `FAIL_ON_MISSING_MODEL_DIRS` is set to `True`
- if the step fails during embedding, check the checkpoint folder layout and the selected tokenizer or experiment filters first


In [ ]:
# Run the full notebook-14 workflow: build embeddings, fit the logistic
# regression probe on the train split, evaluate on each requested split, and
# save the comparison tables, logistic-regression diagnostics, and HTML report.
probe_results = run_embedding_logreg_suite(
    annotated_df=annotated_probe_df,
    run_specs=run_specs,
    checkpoints_dir=CHECKPOINTS_DIR,
    output_paths=output_paths,
    pooling_strategy=POOLING_STRATEGY,
    embedding_layer_index=EMBEDDING_LAYER_INDEX,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    train_splits=TRAIN_SPLITS,
    evaluation_splits=EVALUATION_SPLITS,
    probability_threshold=PROBABILITY_THRESHOLD,
    regularization_c=LOGREG_C,
    class_weight=CLASS_WEIGHT,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    fail_on_missing_model_dirs=FAIL_ON_MISSING_MODEL_DIRS,
    metric_plot_names=METRIC_PLOT_NAMES,
    report_title=HTML_REPORT_TITLE,
)

print(f"Completed runs: {probe_results['metrics_df']['display_label'].nunique()}")
print(f"Skipped runs: {len(probe_results['skipped_df'])}")
print(f"Saved metrics table: {output_paths['split_metrics_path']}")
print(f"Saved test summary: {output_paths['test_summary_path']}")
for plot_name, plot_path in probe_results["diagnostic_plot_paths"].items():
    if plot_path is not None:
        print(f"Saved diagnostic plot [{plot_name}]: {plot_path}")
print(f"HTML report: {probe_results['html_report_path']}")

display(HTML(
    f'<p><a href="{probe_results["html_report_path"]}" target="_blank">Open full HTML logistic-regression report</a></p>'
))

public_export_artifacts = None

if PUBLIC_EXPORT_ENABLED:
    public_export_artifacts = export_public_embedding_logreg_html(
        probe_results=probe_results,
        export_dir=PUBLIC_EXPORT_DIR,
        repo_public_subdir=PUBLIC_EXPORT_REPO_SUBDIR,
        repo_owner=PUBLIC_GITHUB_OWNER,
        repo_name=PUBLIC_GITHUB_REPO,
        repo_ref=PUBLIC_GITHUB_REF,
    )

    print(f'Public export Drive folder: {public_export_artifacts["public_export_dir"]}')
    print(f'Repo folder to copy into before push: {PUBLIC_EXPORT_REPO_SUBDIR}')
    print(f'Repo report path after push: {public_export_artifacts["repo_index_path"]}')
    print(f'GitHack URL after push: {public_export_artifacts["githack_url"]}')
    print('')

    print('=== Copied public files ===')
    display(public_export_artifacts['copied_files_df'])

    print('=== Dependency issues ===')
    if public_export_artifacts['dependency_issues_df'].empty:
        print('No missing local HTML dependencies were found in the public export.')
    else:
        display(public_export_artifacts['dependency_issues_df'])

    print('=== Sensitive-string scan ===')
    if public_export_artifacts['scan_results_df'].empty:
        print('No obvious local Drive paths were found in the copied files.')
    else:
        display(public_export_artifacts['scan_results_df'])

    if public_export_artifacts['has_dependency_issues']:
        raise ValueError(
            'The public export still has missing local dependencies. Fix those before any GitHub copy step.'
        )

    if public_export_artifacts['has_sensitive_matches'] and PUBLIC_EXPORT_FAIL_ON_SENSITIVE_MATCH:
        raise ValueError(
            'The public export still contains suspicious local-environment strings. Review the scan table before sharing the files.'
        )
else:
    print('PUBLIC_EXPORT_ENABLED is False, so the notebook skipped the clean public-export step.')


## Review the summary tables, saved plots, and HTML report

The validation and test diagnostics are usually the fastest place to start. If the ranking changes a lot between train, validation, and test, that is a sign to inspect sample counts and overfitting rather than treating the top row as a final answer.

**Expected output**
- train, validation, and test summary tables for the compared runs
- one saved metric-grid image per split when that split has results
- validation and test ROC, precision-recall, confusion-matrix, and probability-distribution plots
- an optional skipped-run table when some requested checkpoint folders were missing
- a saved HTML report link from the previous step for easier side-by-side review outside the notebook
- when enabled, public-export copy and scan tables that help verify the report is safe to share

**How to interpret the result**
- use the validation table to decide whether a configuration choice looks robust before focusing too heavily on the final test ranking
- if one run looks excellent on train but weak on validation or test, the embedding space may be overfitting the probe rather than separating the classes cleanly


In [ ]:
# Keep the inline review focused on a compact set of metrics that are easy to
# compare across tokenizer family, architecture, and model state.
summary_columns = [
    "display_label",
    "tokenizer_family",
    "architecture_label",
    "model_variant",
    "row_count",
    "positive_count",
    "roc_auc",
    "average_precision",
    "f1",
    "balanced_accuracy",
    "accuracy",
]

print("Train summary")
display(probe_results["train_summary_df"][summary_columns])
print("Validation summary")
display(probe_results["val_summary_df"][summary_columns])
print("Test summary")
display(probe_results["test_summary_df"][summary_columns])

for split_name in ("train", "val", "test"):
    plot_path = probe_results["plot_paths"].get(split_name)
    if plot_path:
        print(f"{split_name.title()} metric grid")
        display(Image(filename=str(plot_path)))

diagnostic_plot_order = [
    ("val_roc", "Validation ROC curve"),
    ("val_pr", "Validation precision-recall curve"),
    ("val_confusion", "Validation confusion matrices"),
    ("val_probability", "Validation probability distributions"),
    ("test_roc", "Test ROC curve"),
    ("test_pr", "Test precision-recall curve"),
    ("test_confusion", "Test confusion matrices"),
    ("test_probability", "Test probability distributions"),
]

for plot_key, plot_title in diagnostic_plot_order:
    plot_path = probe_results["diagnostic_plot_paths"].get(plot_key)
    if plot_path:
        print(plot_title)
        display(Image(filename=str(plot_path)))

if not probe_results["skipped_df"].empty:
    print("Skipped runs")
    display(probe_results["skipped_df"])
